In [ ]:
import logging

from IPython.core.display import Markdown

from library.circuitry import Circuitry
from library.common import Pauli
from library.magic_state_cultivation import MagicStateCultivation
from library.steane_code.patch import SteaneCodePatch
from utils.simulation.clifft import sample

logging.basicConfig(level=logging.ERROR)

In [ ]:
TARGET_DISTANCE = 7

SUPERDENSE_ROUNDS = 3
TELEPORT_ROUNDS = 3
ROUNDS_FOR_COMPLEMENTARY_GAP = 1

if TARGET_DISTANCE % 2 != 1 and TARGET_DISTANCE < 7:
    raise ValueError("TARGET_DISTANCE must be odd and above 7.")

In [ ]:
FILEROOT = "generated/hirano-magic-state-cultivation-layout1"
observables = [Pauli.X, Pauli.Z]
scenarios: dict[str, Circuitry] = dict()
point = 0

In [ ]:
if TARGET_DISTANCE % 2 != 1 and TARGET_DISTANCE < 7:
    raise ValueError("TARGET_DISTANCE must be odd and above 7.")

minimum_anchoring = 1 + int(TARGET_DISTANCE == 7)
TARGET_ANCHOR = (minimum_anchoring, minimum_anchoring)

In [ ]:
for observable in observables:
    msc = MagicStateCultivation(
        injection=SteaneCodePatch.Injection.T, target_distance=TARGET_DISTANCE, anchor=TARGET_ANCHOR
    )

    msc.append_preparation()

    msc.circuitry.append_observable(
        0, f"{observable}_OBSERVABLE_PREPARED", msc.steane.logical(observable)
    )

    scenario = rf"Prepared [$\overline{{\mathbf{{{observable.name}}}}}$]"
    scenarios[scenario] = msc.circuitry

    msc.circuitry.to_file(FILEROOT + f".point{point}.prepared.{observable.name.lower()}-basis")
    display(Markdown(f"[Open in Crumble ({scenario})]({msc.circuitry.to_crumble_url()})"))

point += 1

In [ ]:
for observable in observables:
    msc = MagicStateCultivation(
        injection=SteaneCodePatch.Injection.T, target_distance=TARGET_DISTANCE, anchor=TARGET_ANCHOR
    )

    msc.append_preparation()
    for rnd in range(SUPERDENSE_ROUNDS):
        msc.append_superdense_cycle(rnd)

    msc.annotate_detectors(sdc_rounds=SUPERDENSE_ROUNDS, tpt_rounds=0)
    msc.circuitry.append_observable(0, f"{observable}_OBSERVABLE_SUPERDENSED", msc.steane.logical(observable))

    scenario = rf"SDCx{SUPERDENSE_ROUNDS} [$\overline{{\mathbf{{{observable.name}}}}}$]"
    scenarios[scenario] = msc.circuitry
    msc.circuitry.to_file(FILEROOT + f".point{point}.superdense.{observable.name.lower()}-basis")

    display(Markdown(f"[Open in Crumble ({scenario})]({msc.circuitry.to_crumble_url()})"))

point += 1

In [ ]:
for observable in observables:
    msc = MagicStateCultivation(
        injection=SteaneCodePatch.Injection.T, target_distance=TARGET_DISTANCE, anchor=TARGET_ANCHOR
    )

    msc.append_preparation()
    for rnd in range(SUPERDENSE_ROUNDS):
        msc.append_superdense_cycle(rnd)
    msc.append_cultivation()

    msc.annotate_detectors(sdc_rounds=SUPERDENSE_ROUNDS, tpt_rounds=0)
    msc.circuitry.append_observable(0, f"{observable}_OBSERVABLE_DOUBLE_CHECKED", msc.steane.logical(observable))

    scenario = rf"Double-Check-$\mathbf{{T}}$ [$\overline{{\mathbf{{{observable.name}}}}}$]"
    scenarios[scenario] = msc.circuitry
    msc.circuitry.to_file(FILEROOT + f".point{point}.double-check-t.{observable.name.lower()}-basis")

    display(Markdown(f"[Open in Crumble ({scenario})]({msc.circuitry.to_crumble_url()})"))

point += 1

In [ ]:
post_teleportation_extras: dict[Pauli, list[str]] = {
    Pauli.X : ["STN:TPT0:XB", "STN:TPT1:XB", "STN:TPT2:XB", "STN:DST:X1", "STN:DST:X5", "STN:DST:X6"],
    Pauli.Y : ["JCT0:Z0", "JCT0:Z1", "JCT0:Z2", "STN:TPT0:XB", "STN:TPT1:XB", "STN:TPT2:XB", "STN:DST:X1", "STN:DST:X5", "STN:DST:X6"],
    Pauli.Z : ["JCT0:Z0", "JCT0:Z1", "JCT0:Z2"]
}

In [ ]:
for observable in observables:
    msc = MagicStateCultivation(
        injection=SteaneCodePatch.Injection.T, target_distance=TARGET_DISTANCE, anchor=TARGET_ANCHOR
    )

    msc.append_preparation()
    for rnd in range(SUPERDENSE_ROUNDS):
        msc.append_superdense_cycle(rnd)
    msc.append_cultivation()
    msc.append_teleportation(TELEPORT_ROUNDS)

    msc.annotate_detectors(sdc_rounds=SUPERDENSE_ROUNDS, tpt_rounds=TELEPORT_ROUNDS)

    msc.circuitry.append_observable(
        0, f"{observable}_OBSERVABLE_TELEPORTED", msc.source.logical(observable),
        *post_teleportation_extras[observable], flip=observable == Pauli.Y,
    )

    scenario = rf"Teleported [$\overline{{\mathbf{{{observable.name}}}}}$]"
    scenarios[scenario] = msc.circuitry
    msc.circuitry.to_file(FILEROOT + f".point{point}.teleported.{observable.name.lower()}-basis")

    display(Markdown(f"[Open in Crumble ({scenario})]({msc.circuitry.to_crumble_url()})"))

point += 1

In [ ]:
for observable in observables:
    msc = MagicStateCultivation(
        injection=SteaneCodePatch.Injection.T, target_distance=TARGET_DISTANCE, anchor=TARGET_ANCHOR
    )

    msc.append_preparation()
    for rnd in range(SUPERDENSE_ROUNDS):
        msc.append_superdense_cycle(rnd)
    msc.append_cultivation()
    msc.append_teleportation(TELEPORT_ROUNDS)
    msc.append_expansion()

    msc.annotate_detectors(sdc_rounds=SUPERDENSE_ROUNDS, tpt_rounds=TELEPORT_ROUNDS)

    msc.circuitry.append_observable(
        0, f"{observable}_OBSERVABLE_EXPANDED", msc.target.logical(observable),
        *post_teleportation_extras[observable], flip=observable == Pauli.Y,
    )

    scenario = rf"Expanded [$\overline{{\mathbf{{{observable.name}}}}}$]"
    scenarios[scenario] = msc.circuitry
    msc.circuitry.to_file(FILEROOT + f".point{point}.expanded.{observable.name.lower()}-basis")

    display(Markdown(f"[Open in Crumble ({scenario})]({msc.circuitry.to_crumble_url()})"))

point += 1

In [ ]:
# Analyse error rates of all cumulative circuits
title = r"Magic State Cultivation of $|\mathbf{T}\rangle$ [Corrected $\frac{\overline{\mathbf{X}}+\overline{\mathbf{Z}}}{\sqrt{2}}$]"
sample(
    scenarios,
    title=title,
    label="Point",
    shots=1e6,
    correction=True,
    figsize=(16, 5),
    fontsize=8,
)